# AprendIA · Auditoría y análisis del recorrido educativo

Proyecto de fin de bootcamp sobre una plataforma de aprendizaje simulada. Equipo: Manuela Zapata Jiménez, Astrid Rodríguez y Valerie Avendaño.

**¿Dónde pierde valor el recorrido del estudiante y qué parte de las suscripciones se traduce en ingresos?** Este cuaderno valida el extracto usado para preparar Power BI, revisa el recorrido desde la inscripción hasta la certificación y concilia las suscripciones con los intentos de pago.

[Informe de portafolio](README.md) · [Cinco consultas MySQL del bootcamp](sql/preguntas_negocio_mysql.sql) · [Power BI y DAX](docs/power_bi.md)

## 1. Datos y alcance

El archivo agregado incluido proviene de las 12 hojas de datos de `PowerBI_Aprendia_Paquete.xlsx`. El [script de preparación](preparar_agregados.py) lee las columnas necesarias con Pandas, valida claves y genera el JSON. El cuaderno se ejecuta con el agregado, que contiene conteos y resultados por tabla, plan y categoría.

In [1]:
from pathlib import Path
from decimal import Decimal
import json
import pandas as pd

corte = json.loads(Path('datos/corte_agregado.json').read_text(encoding='utf-8'))
tablas = pd.DataFrame(corte['tablas']).set_index('tabla')
assert len(tablas) == 12 and tablas['filas'].sum() == 51_961
print(tablas.to_string())

                    filas  claves_nulas  claves_repetidas  referencias_huerfanas
tabla                                                                           
plan                    5             0                 0                      0
Usuario              5099             0                 0                      0
Curso                 150             0                 0                      0
Modulo                403             0                 0                      0
Leccion              5281             0                 0                   2663
Material_leccion    10465             0                 0                      0
Inscripcion         12833             0                 0                     59
Suscripcion          5494             0                 0                     23
Pago                 5494             0                 0                      0
Detalle_estudiante   5000             0                 0                     21
Detalle_Profesor      120   

## 2. Conciliación del corte y controles de integridad

La presentación final (págs. 9–10) reportó 52.404 filas, 5.120 usuarios y 825 módulos. El extracto para Power BI contiene 51.961 filas, 5.099 usuarios y 403 módulos. Las diferencias de usuarios y módulos suman exactamente las 443 filas de diferencia entre los totales. La conciliación de conteos no permite comprobar la identidad de cada fila entre ambos cortes.

La verificación distingue claves primarias repetidas o nulas de referencias sin fila correspondiente. En el extracto, las lecciones con módulo ausente afectan la cobertura si se intenta medir cantidad de contenido por curso. Por eso el análisis de conversión de este cuaderno agrupa las inscripciones directamente por categoría de curso.

In [2]:
publicado = {'Usuario': 5_120, 'Modulo': 825}
conciliacion = pd.DataFrame({'presentacion': publicado, 'extracto': tablas['filas'].loc[list(publicado)]})
conciliacion['diferencia'] = conciliacion['presentacion'] - conciliacion['extracto']
assert conciliacion['diferencia'].sum() == 52_404 - tablas['filas'].sum()
assert tablas[['claves_nulas', 'claves_repetidas']].sum().sum() == 0
assert tablas.loc['Leccion', 'referencias_huerfanas'] == 2_663
print(conciliacion.to_string())
print('Lecciones sin módulo correspondiente:', tablas.loc['Leccion', 'referencias_huerfanas'])

         presentacion  extracto  diferencia
Usuario          5120      5099          21
Modulo            825       403         422
Lecciones sin módulo correspondiente: 2663


## 3. Del registro a la certificación

La tasa de finalización divide las inscripciones con estado `completado` por el total de inscripciones. La tasa de certificación divide los certificados por ese mismo total. Las tasas por categoría se calculan a partir de sus conteos y el resultado global pondera por inscripciones.

Las referencias a usuarios ausentes en `Usuario` se mantienen en el denominador de inscripciones. Una lectura por atributos de usuario requiere contar cuántas filas quedan sin perfil. En la duración se excluyen los intervalos negativos del promedio ajustado y se documenta cuántos se encontraron.

In [3]:
aprendizaje = corte['aprendizaje']
categorias = pd.DataFrame(corte['categorias'])
categorias['finalizacion_pct'] = 100 * categorias['finalizadas'] / categorias['inscripciones']
categorias['certificacion_pct'] = 100 * categorias['certificados'] / categorias['inscripciones']
assert categorias['inscripciones'].sum() == aprendizaje['inscripciones'] == 12_833
assert categorias['finalizadas'].sum() == aprendizaje['finalizadas'] == 3_234
assert categorias['certificados'].sum() == aprendizaje['certificados'] == 1_617
assert categorias['fechas_invertidas'].sum() == aprendizaje['fechas_invertidas'] == 1

total = aprendizaje['inscripciones']
print(f"Finalización global: {aprendizaje['finalizadas'] / total:.1%}")
print(f"Certificados por inscripción: {aprendizaje['certificados'] / total:.1%}")
print(f"Días promedio según todas las fechas: {aprendizaje['media_dias_original']:.2f}")
print(f"Días promedio con fechas válidas: {aprendizaje['media_dias_validos']:.2f}")
campos = ['categoria', 'inscripciones', 'finalizacion_pct', 'certificacion_pct']
print(categorias[campos].round(2).to_string(index=False))

Finalización global: 25.2%
Certificados por inscripción: 12.6%
Días promedio según todas las fechas: 70.23
Días promedio con fechas válidas: 70.28
                    categoria  inscripciones  finalizacion_pct  certificacion_pct
Programación y Desarrollo Web           2073             26.24              13.41
          Finanzas Personales           1583             25.90              12.57
           Fotografía y Video           1555             25.53              13.38
            Marketing Digital           1400             25.14              11.21
    Negocios y Emprendimiento           1336             24.70              11.45
          Desarrollo Personal           1279             24.24              12.51
        Ciencia de Datos e IA           1189             25.15              13.37
       Diseño Gráfico y UX/UI           1104             24.28              11.87
                      Idiomas            797             23.09              12.30
               Música y Audio    

## 4. Suscripciones, intentos e ingresos

Las suscripciones se cuentan en su propia tabla. Los intentos y los centavos cobrados se agregan por separado y se unen a la dimensión de planes. Ese orden mantiene el mismo significado de las métricas si existen reintentos o renovaciones. Los montos cobran sentido en la moneda del extracto, USD, y se suman solo para el estado `completado`.

In [4]:
planes = pd.DataFrame(corte['planes'])
assert (planes['completados'] <= planes['intentos']).all()
usd = Decimal(int(planes['centavos_cobrados'].sum())) / 100
assert usd == Decimal('142671.64')
assert planes['intentos'].sum() == 5_494 and planes['completados'].sum() == 5_096

planes['cobrado_usd'] = planes['centavos_cobrados'] / 100
planes['share_ingresos_pct'] = 100 * planes['centavos_cobrados'] / planes['centavos_cobrados'].sum()
planes['share_suscripciones_pct'] = 100 * planes['suscripciones'] / planes['suscripciones'].sum()
print(f'Ingresos cobrados: USD {usd:,.2f}')
print(planes.drop(columns='centavos_cobrados').round(2).to_string(index=False))

Ingresos cobrados: USD 142,671.64
       plan  suscripciones  intentos  completados  cobrado_usd  share_ingresos_pct  share_suscripciones_pct
Empresarial            303       303          276     55197.24               38.69                     5.52
    Premium            882       882          814     40691.86               28.52                    16.05
Profesional           1442      1442         1290     32237.10               22.60                    26.25
     Básico           1607      1607         1456     14545.44               10.20                    29.25
   Gratuito           1260      1260         1260         0.00                0.00                    22.93


## 5. Lectura de negocio

El plan Empresarial lidera los ingresos con una fracción pequeña de las suscripciones. El plan Básico reúne más suscripciones y deja un importe menor por plan en este corte. Para seguimiento conviene mantener visibles tanto el volumen de suscripciones como los ingresos efectivamente cobrados.

En aprendizaje, el volumen de inscripciones, la finalización y la certificación responden preguntas distintas. El tablero original permite explorarlas por categoría. La calidad desigual del enlace entre módulos y lecciones limita la interpretación conjunta de cantidad de contenido y finalización en este extracto.

[Ver capturas originales del tablero y sus medidas](docs/power_bi.md)